## Importing libraries

In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

## splitting the data into trarget and features

In [3]:
df = pd.read_csv("train.csv")
X = df.drop("Heart Disease", axis=1)
y = df["Heart Disease"]

## Basic data preprocessing

In [4]:
X = X.fillna(X.mean(numeric_only=True))
X = pd.get_dummies(X, drop_first=True)

## splitting the data into training and testing

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

## Scaling for the data

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Bagging classifier

In [7]:
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    max_samples=0.8,
    max_features=0.8,
    bootstrap=True,
    random_state=42
)

start = time.time()
bagging.fit(X_train, y_train)
bagging_time = time.time() - start

## AdaBoost classifier

In [8]:
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=200,
    learning_rate=0.5,
    random_state=42
)

start = time.time()
ada.fit(X_train, y_train)
ada_time = time.time() - start

## Gradient boost classifier

In [9]:
gb = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    random_state=42
)

start = time.time()
gb.fit(X_train, y_train)
gb_time = time.time() - start

## Stacking classifier

In [11]:
base_models = [
    ('lr', LogisticRegression(max_iter=200)),
    ('rf', RandomForestClassifier(n_estimators=200)),
    ('gb', GradientBoostingClassifier())
]
stack = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression()
)

start = time.time()
stack.fit(X_train_scaled, y_train)
stack_time = time.time() - start

## Model evaluation

In [12]:
def evaluate_model(model, X_test, y_test):

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    if hasattr(model, "predict_proba"):
        roc = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
    else:
        roc = None
    cm = confusion_matrix(y_test, y_pred)
    return accuracy, precision, recall, f1, roc, cm

bag_results = evaluate_model(bagging, X_test, y_test)
ada_results = evaluate_model(ada, X_test, y_test)
gb_results = evaluate_model(gb, X_test, y_test)
stack_results = evaluate_model(stack, X_test_scaled, y_test)

## Visualizing the comparison

In [13]:
results = pd.DataFrame({
    "Model": ["Bagging", "AdaBoost", "Gradient Boosting", "Stacking"],

    "Accuracy":[
        bag_results[0],
        ada_results[0],
        gb_results[0],
        stack_results[0]
    ],

    "F1 Score":[
        bag_results[3],
        ada_results[3],
        gb_results[3],
        stack_results[3]
    ],

    "ROC AUC":[
        bag_results[4],
        ada_results[4],
        gb_results[4],
        stack_results[4]
    ],

    "Training Time":[
        bagging_time,
        ada_time,
        gb_time,
        stack_time
    ]
})

results

,Model,Accuracy,F1 Score,ROC AUC,Training Time
0,Bagging,0.885317,0.885153,0.951387,279.512820
1,AdaBoost,0.887754,0.887589,0.954362,286.915541
2,Gradient Boosting,0.889135,0.889035,0.955330,270.819089
3,Stacking,0.888167,0.888066,0.954404,1794.042146
